# Notebook 0 — Exploratory Data Analysis (EDA)

Use this notebook before modelling to understand any CSV dataset. It is designed for Google Colab or local Jupyter use: load a CSV, choose columns with GUI controls, and generate an exploratory report without editing code.

Recommended order: run each cell from top to bottom, load or upload your CSV, click **Confirm Selection**, then use the EDA controls.


In [ ]:
# SETUP
import sys, os, subprocess, importlib.util
from pathlib import Path

# Works locally from workshop-2/notebooks, from workshop-2, or in Google Colab
# after opening the notebook from GitHub.
REPO_URL = 'https://github.com/indicium15/ml-workshop.git'

def _find_workshop_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd() / 'workshop-2',
        Path.cwd().parent / 'workshop-2',
        Path('/content/ml-workshop/workshop-2'),
        Path('/content/workshop-2'),
    ]
    for candidate in candidates:
        if (candidate / 'utils' / 'data_loader.py').exists():
            return candidate.resolve()
    return None

IN_COLAB = 'google.colab' in sys.modules
WORKSHOP_ROOT = _find_workshop_root()

if WORKSHOP_ROOT is None and IN_COLAB:
    print('Workshop files not found in Colab. Cloning the workshop repository...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, '/content/ml-workshop'], check=True)
    WORKSHOP_ROOT = _find_workshop_root()

if WORKSHOP_ROOT is None:
    raise FileNotFoundError(
        'Could not find workshop-2/utils. Locally, start Jupyter from the workshop-2 folder. '
        'In Colab, upload the workshop-2 folder or open the notebook from the GitHub repository.'
    )

if str(WORKSHOP_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSHOP_ROOT))

required_modules = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'scipy': 'scipy',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'ipywidgets': 'ipywidgets',
}
missing = [pip_name for module_name, pip_name in required_modules.items()
           if importlib.util.find_spec(module_name) is None]
if missing:
    print('Installing missing packages:', ', '.join(missing))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORKSHOP_ROOT / 'requirements.txt')], check=True)

import ipywidgets as widgets

print(f'Using workshop files from: {WORKSHOP_ROOT}')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns

from utils.data_loader import DataLoaderWidget
from utils.plotting import plot_correlation_heatmap, plot_feature_distributions

print('Libraries loaded ✓')


## Step 1 — Load Data

Use the sample dataset, type a local CSV path, or upload a CSV. Select the columns you want to explore, then click **Confirm Selection**.


In [ ]:
# LOAD DATA
loader = DataLoaderWidget(show_label_selector=False)
loader.display()


## Step 2 — Configure and Run EDA

Click **Refresh Columns** after confirming your data. Then choose the columns, preview size, missing-value threshold, correlation method, and optional grouping column.


In [ ]:
# EDA CONTROLS
_eda_out = widgets.Output()

_preview_rows = widgets.IntSlider(
    value=5, min=3, max=20, step=1,
    description='Preview rows:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='430px'),
)
_missing_threshold = widgets.FloatSlider(
    value=0.0, min=0.0, max=1.0, step=0.05,
    description='Missing >=',
    readout_format='.0%',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='430px'),
)
_max_categories = widgets.IntSlider(
    value=12, min=5, max=30, step=1,
    description='Max categories:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='430px'),
)
_plot_cols = widgets.SelectMultiple(
    options=[],
    value=(),
    description='Plot columns:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='640px', height='220px'),
)
_group_col = widgets.Dropdown(
    options=['(none)'],
    value='(none)',
    description='Group by:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='430px'),
)
_corr_method = widgets.Dropdown(
    options=['pearson', 'spearman', 'kendall'],
    value='pearson',
    description='Correlation:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='300px'),
)
_show_corr = widgets.Checkbox(
    value=True,
    description='Show correlation heatmap',
    layout=widgets.Layout(width='260px'),
)
_show_distributions = widgets.Checkbox(
    value=True,
    description='Show distributions',
    layout=widgets.Layout(width='220px'),
)
_refresh_cols_btn = widgets.Button(
    description='Refresh Columns',
    button_style='',
    icon='refresh',
)
_run_eda_btn = widgets.Button(
    description='Run EDA Report',
    button_style='primary',
    icon='bar-chart',
)

def _active_df():
    if loader.X_df is not None:
        return loader.X_df.copy()
    if loader.df is not None:
        return loader.df.copy()
    return None

def _refresh_eda_columns(_btn=None):
    df = _active_df()
    _eda_out.clear_output()
    if df is None:
        _plot_cols.options = []
        _plot_cols.value = ()
        _group_col.options = ['(none)']
        _group_col.value = '(none)'
        with _eda_out:
            display(widgets.HTML('<span style="color:red">Load and confirm data first.</span>'))
        return

    cols = list(df.columns)
    numeric_cols = list(df.select_dtypes(include='number').columns)
    low_card_cols = [c for c in cols if df[c].nunique(dropna=True) <= _max_categories.value]
    default_cols = (numeric_cols[:8] if numeric_cols else cols[:8])

    _plot_cols.options = cols
    _plot_cols.value = tuple(default_cols)
    group_options = ['(none)'] + low_card_cols
    _group_col.options = group_options
    _group_col.value = '(none)'

    with _eda_out:
        display(widgets.HTML(
            f'<span style="color:green">✓ Ready: {len(df):,} rows × {len(cols)} selected columns. '
            f'{len(numeric_cols)} numeric, {len(cols) - len(numeric_cols)} non-numeric.</span>'
        ))

def _column_overview(df):
    overview = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'non_null': df.notna().sum(),
        'missing': df.isna().sum(),
        'missing_pct': df.isna().mean(),
        'unique_values': df.nunique(dropna=True),
    })
    return overview.sort_values(['missing_pct', 'unique_values'], ascending=[False, False])

def _run_eda(_btn):
    _eda_out.clear_output()
    with _eda_out:
        try:
            df = _active_df()
            if df is None:
                display(widgets.HTML('<span style="color:red">Load and confirm data first.</span>'))
                return
            selected_cols = list(_plot_cols.value) or list(df.columns)[:8]
            selected_cols = [c for c in selected_cols if c in df.columns]
            report_df = df[selected_cols].copy()

            display(widgets.HTML(f'<h3>Dataset Snapshot</h3><b>{len(df):,}</b> rows × <b>{len(df.columns):,}</b> selected columns'))
            display(df.head(_preview_rows.value))

            display(widgets.HTML('<h3>Column Overview</h3>'))
            overview = _column_overview(df)
            display(overview.style.format({'missing_pct': '{:.1%}'}).background_gradient(subset=['missing_pct'], cmap='Reds'))

            missing = overview[overview['missing_pct'] >= _missing_threshold.value]
            display(widgets.HTML(f'<h3>Missing Values ≥ {_missing_threshold.value:.0%}</h3>'))
            if missing.empty:
                display(widgets.HTML('<span style="color:green">No columns meet this missing-value threshold.</span>'))
            else:
                display(missing[['missing', 'missing_pct']].style.format({'missing_pct': '{:.1%}'}).background_gradient(cmap='Reds'))

            numeric_cols = list(report_df.select_dtypes(include='number').columns)
            cat_cols = [c for c in report_df.columns if c not in numeric_cols]

            if numeric_cols:
                display(widgets.HTML('<h3>Numeric Summary</h3>'))
                display(report_df[numeric_cols].describe().T.style.format('{:.2f}'))
            if cat_cols:
                display(widgets.HTML('<h3>Categorical Summary</h3>'))
                cat_summary = []
                for col in cat_cols:
                    counts = report_df[col].astype(str).value_counts(dropna=False).head(_max_categories.value)
                    cat_summary.append({
                        'column': col,
                        'unique_values': report_df[col].nunique(dropna=True),
                        'top_value': counts.index[0] if len(counts) else '',
                        'top_count': int(counts.iloc[0]) if len(counts) else 0,
                    })
                display(pd.DataFrame(cat_summary))

            if _show_distributions.value and selected_cols:
                display(widgets.HTML('<h3>Distributions</h3>'))
                fig = plot_feature_distributions(report_df, selected_cols, ncols=3)
                plt.show()

            if _show_corr.value and len(numeric_cols) >= 2:
                display(widgets.HTML(f'<h3>Correlation Heatmap ({_corr_method.value})</h3>'))
                corr_df = report_df[numeric_cols].corr(method=_corr_method.value)
                fig, ax = plt.subplots(figsize=(max(7, len(numeric_cols) * 0.75), max(5, len(numeric_cols) * 0.55)))
                mask = np.triu(np.ones_like(corr_df, dtype=bool))
                sns.heatmap(corr_df, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn', center=0, linewidths=0.5, ax=ax)
                ax.set_title(f'{_corr_method.value.title()} Correlation')
                plt.tight_layout()
                plt.show()
            elif _show_corr.value:
                display(widgets.HTML('<span style="color:#555">Correlation heatmap needs at least two numeric selected columns.</span>'))

            group_col = _group_col.value
            if group_col != '(none)' and group_col in df.columns and numeric_cols:
                display(widgets.HTML(f'<h3>Group Comparison by <code>{group_col}</code></h3>'))
                grouped = df[[group_col] + numeric_cols].copy()
                if grouped[group_col].nunique(dropna=True) > _max_categories.value:
                    top_groups = grouped[group_col].astype(str).value_counts().head(_max_categories.value).index
                    grouped = grouped[grouped[group_col].astype(str).isin(top_groups)]
                means = grouped.groupby(group_col, dropna=False)[numeric_cols].mean()
                display(means.style.format('{:.2f}').background_gradient(cmap='YlGnBu', axis=0))
                fig, ax = plt.subplots(figsize=(max(8, len(numeric_cols) * 0.7), max(4, len(means) * 0.55)))
                sns.heatmap(means, annot=True, fmt='.1f', cmap='YlGnBu', linewidths=0.5, ax=ax)
                ax.set_title(f'Mean numeric values by {group_col}')
                plt.tight_layout()
                plt.show()

        except Exception as exc:
            import traceback
            display(widgets.HTML(f'<span style="color:red">Error: {exc}<br><pre>{traceback.format_exc()}</pre></span>'))

_refresh_cols_btn.on_click(_refresh_eda_columns)
_run_eda_btn.on_click(_run_eda)

display(widgets.VBox([
    widgets.HTML('<h3>🔎 EDA Controls</h3>'),
    widgets.HBox([_refresh_cols_btn, _run_eda_btn]),
    widgets.HBox([_preview_rows, _missing_threshold]),
    widgets.HBox([_max_categories, _corr_method]),
    widgets.HBox([_show_distributions, _show_corr]),
    _group_col,
    _plot_cols,
    _eda_out,
]))


## Reflection

Use the EDA report to decide which columns are suitable for clustering or prediction, which columns need cleaning, whether outliers or missing values need attention, and whether any columns should be excluded before modelling.
